### Jiwei + Eduardo 0.955

In [1]:
"""Private offline B1 plus B1-label C1 inference on two T4 GPUs."""

from __future__ import annotations

import hashlib
import gc
import importlib.util
import json
import os
from pathlib import Path
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch

# Kaggle has few CPU cores. Bound intra-op parallelism when several DICOM
# decoding tasks each resize tensors; otherwise nested thread pools contend.
torch.set_num_threads(4)


TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
C1_WEIGHTS = np.asarray([0.2, 0.2, 0.2, 0.3, 0.4, 0.5, 0.5, 0.1, 0.3, 0.2, 0.3, 0.3])
ENCODER_SHA256 = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
WORK = Path(os.environ.get("RSNA_WORK_DIR", "/kaggle/working"))


def sha256_file(path: Path, chunk_size: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def import_file(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def find_unique(manifest_name: str, local: Path) -> Path:
    slugs = {
        "b1_maxspan_manifest.json": "rsna-knee-b1-maxspan-all-data-assets-v1",
        "b1_c1_manifest.json": "rsna-knee-b1-c1-all-data-assets-v1",
    }
    slug = slugs[manifest_name]
    roots = [Path(f"/kaggle/input/datasets/jiweiliu/{slug}"),
             Path(f"/kaggle/input/{slug}")]
    matches = sorted({(root / manifest_name).resolve() for root in roots
                      if (root / manifest_name).is_file()})
    if not matches and local.is_file():
        matches = [local]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected one {manifest_name}, observed {len(matches)}: {matches}"
        )
    return matches[0].parent


def find_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("input/rsna-knee-abnormality-detection"),
    ]
    matches = [root for root in candidates if (root / "test.csv").is_file()]
    unique = {str(root.resolve()): root for root in matches}
    if len(unique) != 1:
        raise FileNotFoundError(f"expected one competition input, observed {matches}")
    return next(iter(unique.values()))


def find_encoder() -> Path:
    configured = os.environ.get("RSNA_RADIMAGENET_ENCODER")
    roots = [Path("/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan"),
             Path("/kaggle/input/resnet-50-radimagenet-marwan")]
    candidates = [Path(configured)] if configured else (
        [path for root in roots if root.is_dir() for path in root.rglob("ResNet50.pt")]
        + [Path("models/radimagenet-encoder/ResNet50.pt")]
    )
    matches = sorted({
        path.resolve() for path in candidates
        if path.is_file() and sha256_file(path) == ENCODER_SHA256
    })
    if len(matches) != 1:
        raise FileNotFoundError(f"expected one pinned RadImageNet encoder, observed {len(matches)}")
    return matches[0]


def rank_columns(values: np.ndarray) -> np.ndarray:
    return pd.DataFrame(values).rank(method="average", pct=True).to_numpy(float)


def validate_submission(frame: pd.DataFrame, ids: list[str]) -> None:
    if frame.columns.tolist() != ["StudyInstanceUID", *TARGETS]:
        raise RuntimeError("submission schema drift")
    if frame.StudyInstanceUID.astype(str).tolist() != ids:
        raise RuntimeError("submission identity/order drift")
    values = frame[TARGETS].to_numpy(float)
    if not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError("submission value-domain drift")


def predict_source_pipelined(root, test, package, devices, log, runtime):
    """Bounded decode-ahead and independent ordered queues for both GPUs."""
    started = time.monotonic()
    if len(devices) != 2 or torch.cuda.device_count() != 2:
        raise RuntimeError("B1 requires exactly two GPUs")
    manifest = runtime.load_package(package)
    core = runtime.load_core(package)
    setup_started = time.monotonic()
    groups = {device: [] for device in devices}
    for member in range(runtime.N_MEMBERS):
        device = devices[member % 2]
        groups[device].append(
            (member, runtime.load_model(package, member, manifest, core, device))
        )
    setup_seconds = time.monotonic() - setup_started
    series = pd.read_csv(root / "test_series.csv", dtype={
        "StudyInstanceUID": str, "SeriesInstanceUID": str,
    })
    uids = test.StudyInstanceUID.astype(str).tolist()
    series = series.loc[series.StudyInstanceUID.isin(set(uids))]
    records = {uid: frame.to_dict("records") for uid, frame in
               series.groupby("StudyInstanceUID", sort=False)}
    series_root = root / "test_series"
    if not series_root.is_dir():
        series_root = root / "test_images"
    raw = np.full((runtime.N_MEMBERS, len(test), len(TARGETS)), np.nan, np.float32)
    decode_wait = gpu_wait = 0.0

    def prepare(uid):
        volume, mask = core.build_study(uid, records.get(uid, []), series_root)
        return core.eval_windows(volume, mask)

    def collect(item):
        nonlocal gpu_wait
        row, futures = item
        before = time.monotonic()
        for future in futures:
            for member, value in future.result():
                raw[member, row] = value
        gpu_wait += time.monotonic() - before

    # One executor per GPU ensures only one study is resident in its forward
    # pass; pending inputs and decoded host tensors are bounded independently
    # of hidden-test size. Exceptions propagate instead of filling predictions.
    inference_started = time.monotonic()
    with ThreadPoolExecutor(max_workers=2) as decode, \
            ThreadPoolExecutor(max_workers=1) as gpu0, \
            ThreadPoolExecutor(max_workers=1) as gpu1:
        prepared, pending = deque(), deque()
        next_row = 0
        while next_row < min(2, len(uids)):
            prepared.append((next_row, decode.submit(prepare, uids[next_row])))
            next_row += 1
        while prepared:
            row, future = prepared.popleft()
            before = time.monotonic()
            windows = future.result()
            decode_wait += time.monotonic() - before
            if next_row < len(uids):
                prepared.append((next_row, decode.submit(prepare, uids[next_row])))
                next_row += 1
            pending.append((row, [pool.submit(core.infer_device,
                groups[device], windows, device)
                for pool, device in zip((gpu0, gpu1), devices)]))
            if len(pending) >= 2:
                collect(pending.popleft())
            if (row + 1) % 25 == 0 or row + 1 == len(uids):
                log(f"B1 pipeline queued {row + 1}/{len(uids)}")
        while pending:
            collect(pending.popleft())
    inference_seconds = time.monotonic() - inference_started
    if not np.isfinite(raw).all():
        raise RuntimeError("missing/non-finite B1 member prediction")
    ranks = np.stack([runtime.rank_columns(values) for values in raw])
    prediction = ranks.mean(axis=0)
    memory = {str(device): torch.cuda.max_memory_reserved(device) for device in devices}
    groups.clear()
    runtime.gc.collect()
    for device in devices:
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
    return prediction, {
        "status": "COMPLETE", "candidate_id": runtime.CANDIDATE_ID,
        "members": runtime.N_MEMBERS, "test_studies": len(test),
        "member_gpu_split": {str(devices[0]): 3, str(devices[1]): 2},
        "member_aggregation": manifest["aggregation"], "failures": [],
        "asset_version": manifest["version"],
        "wall_seconds": time.monotonic() - started,
        "model_setup_seconds": setup_seconds,
        "inference_seconds": inference_seconds,
        "decode_blocking_seconds": decode_wait, "gpu_wait_seconds": gpu_wait,
        "peak_reserved_bytes": memory, "decode_workers": 2,
        "pending_studies": 2,
    }


def extract_dense_features_pipelined(runtime, root, test, encoder_path, log):
    """Retain original 16-study blocks and 384-image batches on two GPUs."""
    started = time.monotonic()
    devices = [torch.device("cuda:0"), torch.device("cuda:1")]
    state = torch.load(encoder_path, map_location="cpu", weights_only=True)
    encoders = []
    for device in devices:
        encoder = runtime.RadEncoder()
        encoder.load_state_dict(state, strict=True)
        encoders.append(encoder.eval().to(device))
    del state
    selected, availability = runtime._select_dense_series(root, test)
    if availability["overall_fill"] < 0.60:
        raise RuntimeError("dense test slot fill unexpectedly low")
    features = np.zeros((len(test), len(runtime.DENSE_SLOTS),
                         runtime.DENSE_SLICES, runtime.DENSE_DIM), np.float16)
    masks = np.zeros((len(test), len(runtime.DENSE_SLOTS)), np.uint8)
    ordering = {"geometry": 0, "instance": 0, "filename": 0}
    unresolved = 0
    decode_wait = encode_seconds = 0.0

    def submit_block(pool, start):
        stop = min(start + 16, len(test))
        jobs = [(study, slot, record) for study in range(start, stop)
                for slot, record in enumerate(selected[study]) if record is not None]
        return stop, jobs, [pool.submit(runtime._decode_dense, job[2]) for job in jobs]

    def encode(index, pixels):
        device = devices[index]
        torch.cuda.set_device(device)
        with torch.inference_mode():
            image = torch.from_numpy(pixels).to(device).float()
            image = image.div_(127.5).sub_(1.0)
            image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
            with torch.autocast("cuda", dtype=torch.float16):
                return encoders[index](image).float().cpu().numpy().astype(np.float16)

    try:
        with ThreadPoolExecutor(max_workers=2) as decode, \
                ThreadPoolExecutor(max_workers=1) as gpu0, \
                ThreadPoolExecutor(max_workers=1) as gpu1:
            pending = deque()
            next_start = 0
            while next_start < min(32, len(test)):
                pending.append(submit_block(decode, next_start))
                next_start += 16
            while pending:
                stop, jobs, futures = pending.popleft()
                before = time.monotonic()
                decoded = [future.result() for future in futures]
                decode_wait += time.monotonic() - before
                if next_start < len(test):
                    pending.append(submit_block(decode, next_start))
                    next_start += 16
                volumes, positions = [], []
                for (study, slot, _), (volume, side, source) in zip(jobs, decoded):
                    ordering[source] += 1
                    unresolved += int(side is None)
                    if volume is None:
                        continue
                    masks[study, slot] = 1
                    volumes.append(volume)
                    positions.extend((study, slot, pos) for pos in range(runtime.DENSE_SLICES))
                if volumes:
                    flat = np.concatenate(volumes, axis=0)
                    before = time.monotonic()
                    # At most two active encoder batches. Their boundaries are
                    # exactly the reference boundaries, including the tail.
                    for pair_start in range(0, len(flat), 768):
                        batches = [(offset, pool.submit(encode, index, flat[offset:offset + 384]))
                                   for index, pool in enumerate((gpu0, gpu1))
                                   for offset in [pair_start + index * 384] if offset < len(flat)]
                        for offset, future in batches:
                            values = future.result()
                            study, slot, pos = zip(*positions[offset:offset + len(values)])
                            features[study, slot, pos] = values
                    encode_seconds += time.monotonic() - before
                log(f"dense dual-GPU features {stop}/{len(test)}")
    finally:
        encoders.clear()
        del encoder
        gc.collect()
        for device in devices:
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
    if not masks.any(axis=1).all() or not np.isfinite(features).all():
        raise RuntimeError("dense feature contract failed")
    return features, masks, {
        "availability": availability, "decoded_slot_fill": float(masks.mean()),
        "ordering": ordering, "unresolved_laterality": unresolved,
        "decode_workers": 2, "prefetch_blocks": 2, "block_studies": 16,
        "encoder_batch_images": 384, "devices": [str(d) for d in devices],
        "decode_wait_seconds": decode_wait, "encode_seconds": encode_seconds,
        "wall_seconds": time.monotonic() - started,
    }


def main() -> dict:
    started = time.monotonic()
    root = find_root()
    v5_package = find_unique(
        "b1_maxspan_manifest.json",
        Path("kaggle_assets/b1_maxspan_all_data_v1/b1_maxspan_manifest.json"),
    )
    c1_package = find_unique(
        "b1_c1_manifest.json",
        Path("kaggle_assets/b1_c1_all_data_v1/b1_c1_manifest.json"),
    )
    competition_files = {
        "test_csv": (root / "test.csv").is_file(),
        "test_series_csv": (root / "test_series.csv").is_file(),
        "test_pixels": (root / "test_series").is_dir() or (root / "test_images").is_dir(),
    }
    required_v5 = [
        "b1_maxspan_runtime.py", "maxspan_core.py",
        "b1_maxspan_manifest.json", *[f"member{i}.pth" for i in range(5)],
    ]
    required_c1 = [
        "b1_c1_manifest.json", "b1_c1_dense_sequence_runtime.py",
        "p1_four_source_runtime.py",
        *[
            f"sequence_seed{seed}_replicate{replicate}.pth"
            for seed in (81221, 85621, 90121) for replicate in range(5)
        ],
    ]
    missing = [
        str(path) for path in
        [*[v5_package / name for name in required_v5], *[c1_package / name for name in required_c1]]
        if not path.is_file()
    ]
    preflight = {
        "status": "PASS" if torch.cuda.device_count() == 2 and all(competition_files.values()) and not missing else "FAIL",
        "stage": "startup_input_preflight",
        "gpu_count": torch.cuda.device_count(),
        "competition_root": str(root),
        "v5_package": str(v5_package),
        "c1_package": str(c1_package),
        "competition_files": competition_files,
        "missing": missing,
    }
    print(json.dumps(preflight, indent=2), flush=True)
    if preflight["status"] != "PASS":
        raise RuntimeError("startup input preflight failed")

    c1_runtime = import_file("_b1_c1_runtime", c1_package / "b1_c1_dense_sequence_runtime.py")
    c1_manifest = c1_runtime.validate_package(c1_package, TARGETS)
    v5_runtime = import_file("_b1_runtime", v5_package / "b1_maxspan_runtime.py")
    v5_manifest = v5_runtime.load_package(v5_package)
    feature_runtime = import_file("_b1_c1_feature_runtime", c1_package / "p1_four_source_runtime.py")
    encoder = find_encoder()
    if c1_manifest["formula"]["per_target_c1_weight"] != C1_WEIGHTS.tolist():
        raise RuntimeError("B1-C1 frozen blend weight drift")
    test = pd.read_csv(root / "test.csv", dtype={"StudyInstanceUID": str})
    limit = int(os.environ.get("RSNA_LIMIT", "0"))
    if limit:
        test = test.iloc[:limit].reset_index(drop=True)
    ids = test.StudyInstanceUID.astype(str).tolist()
    print(json.dumps({
        "event": "startup_preflight_pass", "test_studies": len(test),
        "v5_members": v5_manifest["members"], "c1_members": c1_manifest["members"],
    }), flush=True)

    parent_prediction, parent_audit = predict_source_pipelined(
        root, test, v5_package,
        [torch.device("cuda:0"), torch.device("cuda:1")],
        lambda message: print(f"[B1] {message}", flush=True),
        v5_runtime,
    )
    features, masks, extraction = extract_dense_features_pipelined(
        feature_runtime, root, test, encoder,
        lambda message: print(f"[B1-C1] {message}", flush=True),
    )
    c1_prediction, c1_audit = c1_runtime.predict_source(
        c1_package, features, masks, TARGETS, rank_columns, torch.device("cuda:0"),
    )
    prediction = (1 - C1_WEIGHTS[None]) * parent_prediction + C1_WEIGHTS[None] * c1_prediction
    submission = pd.DataFrame(prediction, columns=TARGETS)
    submission.insert(0, "StudyInstanceUID", ids)
    validate_submission(submission, ids)
    WORK.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(WORK / "b1_c1_source_predictions.npz", ids=np.asarray(ids),
                        parent=parent_prediction, c1=c1_prediction, weights=C1_WEIGHTS)
    output = WORK / "submission.csv"
    submission.to_csv(output, index=False)
    validate_submission(pd.read_csv(output, dtype={"StudyInstanceUID": str}), ids)
    audit = {
        "status": "B1_C1_COMPLETE",
        "candidate_id": c1_manifest["candidate_id"],
        "test_studies": len(test),
        "formula": "per-target (1-w)*B1 + w*C1 after source-wise member rank averaging",
        "c1_weights": C1_WEIGHTS.tolist(),
        "parent": parent_audit,
        "dense_extraction": extraction,
        "c1": c1_audit,
        "submission_sha256": sha256_file(output),
        "torch_cpu_threads": torch.get_num_threads(),
        "peak_reserved_bytes": {f"cuda:{i}": torch.cuda.max_memory_reserved(i) for i in range(2)},
        "wall_seconds": time.monotonic() - started,
    }
    (WORK / "b1_c1_runtime_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
    print(json.dumps(audit, indent=2), flush=True)
    return audit


if __name__ == "__main__":
    main()


# Synovitis specialist blend. The complete Jiwei source above is unchanged.
if __name__ == "__main__":
    import subprocess
    import sys
    import zipfile
    VARIANT = 'threeway'
    _dataset_slug = "rsna-knee-syn-specialists-20261009"
    _candidates = [Path("/kaggle/input/datasets/arc144") / _dataset_slug,
                   Path("/kaggle/input") / _dataset_slug]
    _expanded = sorted({p.resolve() for p in _candidates if (p / "manifest.json").is_file()})
    if len(_expanded) == 1:
        # Kaggle automatically extracts uploaded ZIP datasets.
        _assets = _expanded[0]
    else:
        _archives = sorted({(p / "artifacts.zip").resolve() for p in _candidates if (p / "artifacts.zip").is_file()})
        if _expanded or len(_archives) != 1:
            raise RuntimeError(f"Expected one specialist attachment, found {_expanded}, {_archives}")
        _assets = Path("/kaggle/temp/synovitis_specialists")
        _assets.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(_archives[0]) as _zip:
            if any(Path(n).is_absolute() or ".." in Path(n).parts for n in _zip.namelist()):
                raise RuntimeError("Invalid archive member")
            _zip.extractall(_assets)
    _manifest = json.loads((_assets / "manifest.json").read_text())
    for _relative, _expected in _manifest["runtime_sha256"].items():
        if sha256_file(_assets / _relative) != _expected:
            raise RuntimeError("Specialist runtime hash differs: " + _relative)
    _runtime = _assets / "runtime"
    _dependencies = Path("/kaggle/temp/synovitis_dependencies")
    # Install codecs in an isolated subprocess after the untouched original
    # inference has finished. Specialist workers start with these packages.
    _bootstrap = "import json,sys; from pathlib import Path; sys.path.insert(0,sys.argv[1]); from dicom_bootstrap import install_dicom_runtime; r=install_dicom_runtime(sys.argv[2],sys.argv[3]); Path(sys.argv[4]).write_text(json.dumps(r,indent=2))"
    subprocess.run([sys.executable, "-c", _bootstrap, str(_runtime), str(_assets),
                    str(_dependencies), str(WORK / "specialist_codec_environment.json")], check=True)
    _baseline_path = WORK / "baseline_submission.csv"
    _baseline_path.write_bytes((WORK / "submission.csv").read_bytes())
    _baseline = pd.read_csv(_baseline_path, dtype={"StudyInstanceUID": str}, float_precision="round_trip")
    _ids = _baseline.StudyInstanceUID.tolist()
    validate_submission(_baseline, _ids)
    _ids_path = WORK / "specialist_ids.csv"
    _baseline[["StudyInstanceUID"]].to_csv(_ids_path, index=False)
    _env = dict(os.environ, PYTHONPATH=str(_dependencies)+os.pathsep+str(_runtime)+os.pathsep+os.environ.get("PYTHONPATH", ""),
                OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
    _out = WORK / "specialists"
    subprocess.run([sys.executable, str(_runtime / "synovitis_submission_runtime.py"),
        "--bundle", str(_assets), "--root", str(find_root()), "--ids", str(_ids_path),
        "--output", str(_out), "--variant", VARIANT], env=_env, check=True)
    _ours = pd.read_csv(_out / "specialist_predictions.csv", dtype={"StudyInstanceUID": str}, float_precision="round_trip")
    if not _ours.StudyInstanceUID.is_unique or set(_ours.StudyInstanceUID) != set(_ids):
        raise RuntimeError("Specialist study identities differ")
    _ours = _ours.set_index("StudyInstanceUID").loc[_ids]
    _families = ["resnet", "inception", "dino"] if VARIANT == "threeway" else ["resnet"]
    _probabilities = _ours[_families].mean(axis=1).to_numpy()
    if not np.isfinite(_probabilities).all() or (_probabilities < 0).any() or (_probabilities > 1).any():
        raise RuntimeError("Invalid specialist probabilities")
    if not np.allclose(_probabilities, _ours.ours.to_numpy(), atol=1e-14, rtol=0):
        raise RuntimeError("Within-specialist probability average differs")
    _jiwei_rank = pd.Series(_baseline.Synovitis.to_numpy()).rank(method="average", pct=True).to_numpy()
    _ours_rank = pd.Series(_probabilities).rank(method="average", pct=True).to_numpy()
    _submission = _baseline.copy()
    _submission["Synovitis"] = 0.5 * _jiwei_rank + 0.5 * _ours_rank
    validate_submission(_submission, _ids)
    _submission.to_csv(WORK / "submission.csv", index=False)
    _reloaded = pd.read_csv(WORK / "submission.csv", dtype={"StudyInstanceUID": str}, float_precision="round_trip")
    validate_submission(_reloaded, _ids)
    _other = [c for c in TARGETS if c != "Synovitis"]
    if not np.array_equal(_reloaded[_other].to_numpy(), _baseline[_other].to_numpy()):
        raise RuntimeError("A non-synovitis prediction changed")
    if not np.array_equal(_reloaded.Synovitis.to_numpy(), _submission.Synovitis.to_numpy()):
        raise RuntimeError("Synovitis CSV roundtrip differs")
    pd.DataFrame({"StudyInstanceUID": _ids, "jiwei_synovitis": _baseline.Synovitis.to_numpy(),
        "ours_probability": _probabilities, "jiwei_rank": _jiwei_rank, "ours_rank": _ours_rank,
        "final_synovitis": _submission.Synovitis.to_numpy()}).to_csv(WORK / "synovitis_blend_details.csv", index=False)
    _audit = dict(status="SYNOVITIS_BLEND_COMPLETE", variant=VARIANT, rows=len(_ids),
        formula="0.5*rank(original Jiwei Synovitis)+0.5*rank(equal specialist-family probability mean)",
        specialist_families=_families, jiwei_internal_synovitis_weights=dict(B1=0.7,C1=0.3),
        other_eleven_classes_exact=True, baseline_sha256=sha256_file(_baseline_path),
        submission_sha256=sha256_file(WORK / "submission.csv"),
        specialist_manifest_sha256=sha256_file(_assets / "manifest.json"),
        original_script_version_id=356089673, original_source_sha256="ab144618e1c58568e7027f92309864c06cc32a91a73e25ceede2b28c21b65b86")
    (WORK / "synovitis_blend_audit.json").write_text(json.dumps(_audit, indent=2)+"\n")
    print(json.dumps(_audit, indent=2), flush=True)

{
  "status": "PASS",
  "stage": "startup_input_preflight",
  "gpu_count": 2,
  "competition_root": "/kaggle/input/competitions/rsna-knee-abnormality-detection",
  "v5_package": "/kaggle/input/datasets/jiweiliu/rsna-knee-b1-maxspan-all-data-assets-v1",
  "c1_package": "/kaggle/input/datasets/jiweiliu/rsna-knee-b1-c1-all-data-assets-v1",
  "competition_files": {
    "test_csv": true,
    "test_series_csv": true,
    "test_pixels": true
  },
  "missing": []
}
{"event": "startup_preflight_pass", "test_studies": 3, "v5_members": 5, "c1_members": 15}
[B1] B1 pipeline queued 3/3
[B1-C1] dense dual-GPU features 3/3
{
  "status": "B1_C1_COMPLETE",
  "candidate_id": "B1-C1-dense-sequence",
  "test_studies": 3,
  "formula": "per-target (1-w)*B1 + w*C1 after source-wise member rank averaging",
  "c1_weights": [
    0.2,
    0.2,
    0.2,
    0.3,
    0.4,
    0.5,
    0.5,
    0.1,
    0.3,
    0.2,
    0.3,
    0.3
  ],
  "parent": {
    "status": "COMPLETE",
    "candidate_id": "B1-maxspan",
  

## Theo's crops

| Finding           | Models                                                         | 58 / holdout        | dataset_2_moaks_mrnet (previous best) |
| ----------------- | -------------------------------------------------------------- | ------------------- | --------------------------------- |
| ACL               | `acl` + MRNet (weight 2) + `all`                               | 0.983 / 0.994       | 0.983 / 0.994                     |
| MCL               | `mcl` + MRNet pseudo-labels + `all`                            | 0.996 / 0.959       | 0.996 / 0.959                     |
| Medial meniscus   | `menisci` (MOAKS) + CAFormer (sag + cor + axial) + `all`       | **0.983 / 0.955**   | 0.978 / 0.952                     |
| Lateral meniscus  | `menisci` (MOAKS) + CAFormer (read on sag + cor only) + `all`  | **0.959** / 0.920   | 0.945 / 0.920                     |
| Medial OA         | `all` + `oa` 224 px (MOAKS) + `oa` + axial (MOAKS)             | 0.977 / 0.949       | 0.977 / 0.949                     |
| Lateral OA        | `oa` 224 px (MOAKS) only                                       | 0.878 / 0.900       | 0.878 / 0.900                     |
| PF OA             | `oa` + axial (MOAKS) only                                      | 0.923 / 0.937       | 0.923 / 0.937                     |
| Baker's           | `bakers` + `all`                                               | 0.991 / 0.971       | 0.991 / 0.971                     |
| Fracture          | 9-ROI fracture models (fr9a + fr9b, fibular-head crop), **rank-blended 50/50 with b1_c1** | 0.967 / -  (b1_c1 alone 0.953, models alone 0.961 / 0.952) | b1_c1 only |
| **Macro (8 ROI)** |                                                                | **0.9611 / 0.9480** | 0.9588 / 0.9477                   |

dataset_2_moaks_mrnet (submitted, LB +0.0001 over dataset_2_moaks) with two menisci changes; every other model and
recipe entry is unchanged:

1. `mk87` replaces the menisci model `87` (fullfit 2026-10-11/139): the same CoAtNet-0 224 px menisci model (run 81
   setup), initialised from a CoAtNet MOAKS pretraining on OAI (2026-10-11/29, the 25 MOAKS gradings) instead of
   ImageNet. Recipe-level macro over 3 seeds 0.9599 / 0.9591 / 0.9594 (58) vs 0.9588 / 0.9585 for two seeds of the
   original; mostly the lateral meniscus.
2. `cafax87` is added as the third model of both menisci (fullfit 2026-10-12/11): a CAFormer-S18 menisci model trained
   on sagittal + coronal + axial crops (setup of validation run 2026-10-10/93, seed 0, no external data). The lateral
   meniscus uses it as `cafax87@Sagittal,Coronal`, averaged over its sagittal and coronal series only: its axial
   lateral-meniscus predictions are weak (with all planes the lateral meniscus does not gain). Seed check
   (src/ext/men_eval.py, mean over 5 base menisci seeds, mean of both menisci on the 58): base 0.9631, CAFormer seed 0
   0.9698, seed 1 0.9657, with MRNet 0.9654 / 0.9664 (not used: plain is better on the 58). The seed-0 gain is partly
   luck; expect about +0.003 on the menisci. Every run improves both menisci on the 58 and the holdout.

In [2]:
import os
import sys
import json
import time

import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # Code and wheels come from attached datasets; the competition image lacks
    # dynamic_network_architectures (the ROI U-Net) and the DICOM decoders.
    !cp -r /kaggle/input/datasets/theoviel/rsna-knee-code/src ./
    !pip install --quiet --no-deps /kaggle/input/notebooks/theoviel/rsna-knee-wheels/dynamic_network_architectures-0.4.4-py3-none-any.whl
    SRC = os.path.abspath("./src")
else:
    SRC = os.path.abspath("../src")

# The pipeline runs as `<this python> -m infer.pipeline` with src/ on the path.
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, SRC)
print("src:", SRC, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      "| CPUs:", os.cpu_count())

src: /kaggle/working/src | GPUs: ['Tesla T4', 'Tesla T4'] | CPUs: 4


In [3]:
if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
    WEIGHTS = "/kaggle/input/datasets/theoviel/rsna-knee-weights-2-moaks-mrnet-men-frac/"   # output/dataset_2_moaks_mrnet_men_frac
    OUT_DIR = "/tmp/knee/"
else:
    DATA_DIR = "../input/"
    WEIGHTS = "../output/dataset_2_moaks_mrnet_men_frac/"
    OUT_DIR = "../output/inference/"
# WEIGHTS: side/ and roi/ run folders, crops/<run>/ (args.json + ckpt_final.pt), recipe.json

SERIES_DIR = "test_series"                       # <SERIES_DIR>/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
SERIES_CSV = os.path.join(DATA_DIR, "test_series.csv")   # series -> study, plane (folders walked if absent)
GPUS = list(range(torch.cuda.device_count()))    # one process per GPU
WORKERS = max(1, os.cpu_count() // len(GPUS))    # DICOM reading processes per GPU
LIMIT = None                                     # first N studies only (debugging)

recipe = json.load(open(os.path.join(WEIGHTS, "recipe.json")))
models = {m.split("@")[0] for v in recipe.values() for m in (v if isinstance(v, list) else v["models"])}   # "name@planes": plane-restricted use of model name
for w in ("side", "roi") + tuple(f"crops/{m}" for m in models):
    assert os.path.exists(os.path.join(WEIGHTS, w, "ckpt_final.pt")), f"missing weights: {w}"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_DIR, WEIGHTS, OUT_DIR = map(os.path.abspath, (DATA_DIR, WEIGHTS, OUT_DIR))
print("recipe:", recipe)

recipe: {'ACL': ['mr85', '88'], 'MCL': ['pl86', '88'], 'Medial Meniscus': ['mk87', 'cafax87', '88'], 'Lateral Meniscus': ['mk87', 'cafax87@Sagittal,Coronal', '88'], 'Medial OA': ['88', 'mk89', 'mk90'], 'Lateral OA': ['mk89'], 'PF OA': ['mk90'], "Baker's": ['88', '91']}


In [4]:
%%time
cmd = (f"{sys.executable} -m infer.pipeline --data_dir {DATA_DIR} --series_dir {SERIES_DIR} --series_csv {SERIES_CSV}"
       f" --weights {WEIGHTS} --out_dir {OUT_DIR} --submission submission_theo.csv"
       f" --gpus {','.join(map(str, GPUS))} --workers {WORKERS}" + (f" --limit {LIMIT}" if LIMIT else ""))

print(cmd, "\n")
!{cmd}

/usr/bin/python3 -m infer.pipeline --data_dir /kaggle/input/competitions/rsna-knee-abnormality-detection --series_dir test_series --series_csv /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv --weights /kaggle/input/datasets/theoviel/rsna-knee-weights-2-moaks-mrnet-men --out_dir /tmp/knee --submission submission_theo.csv --gpus 0,1 --workers 2 

shard done: 2 studies in 41s -- load 17s, warmup 18s, wait 1s, prep 0s, side 0s, unet 1s, boxes 0s, consensus 0s, crops 0s, models 3s, model 88 0s, model 91 0s, model cafax87 0s, model mk87 1s, model mk89 1s, model mk90 0s, model mr85 0s, model pl86 0s
shard done: 1 studies in 42s -- load 17s, warmup 18s, wait 2s, prep 0s, side 0s, unet 1s, boxes 0s, consensus 0s, crops 0s, models 3s, model 88 0s, model 91 0s, model cafax87 0s, model mk87 1s, model mk89 1s, model mk90 0s, model mr85 0s, model pl86 0s
-> submission_theo.csv: 3 studies, recipe covers ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lat

In [5]:
shards = sorted(d for d in os.listdir(OUT_DIR) if d.startswith("shard") and os.path.exists(os.path.join(OUT_DIR, d, "series.csv")))
series = pd.concat([pd.read_csv(os.path.join(OUT_DIR, d, "series.csv")) for d in shards], ignore_index=True)
rois = pd.concat([pd.read_csv(os.path.join(OUT_DIR, d, "rois.csv")) for d in shards], ignore_index=True)
print(series.plane.value_counts().to_dict(), "| failed:", int((series.status != "ok").sum()),
      "| sides:", series.side.value_counts().to_dict())
print(len(rois), "boxes:", rois.plane.value_counts().to_dict())

sub_theo = pd.read_csv("submission_theo.csv")
sub_theo.head()

{'Sagittal': 7, 'Axial': 4, 'Coronal': 4} | failed: 0 | sides: {'R': 10, 'L': 5}
120 boxes: {'Sagittal': 56, 'Coronal': 32, 'Axial': 32}


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.333333,1.000000,1.000000,0.333333,0.777778,0.082823,0.207248,0.5,0.5,1.000000,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.833333,0.333333,0.666667,0.888889,0.333333,0.433517,0.726805,0.5,0.5,0.333333,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.833333,0.666667,0.333333,0.777778,0.888889,0.279636,0.467260,0.5,0.5,0.666667,0.5,0.5


## Merge

In [6]:
sub = pd.read_csv("submission.csv")
sub.head()

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.466667,0.804444,0.933333,0.72,0.586667,0.333333,0.333333,0.693333,0.666667,1.000000,0.333333,0.413333
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.880000,0.755556,0.733333,0.62,0.413333,0.988889,0.900000,0.940000,1.000000,0.613333,1.000000,1.000000
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.653333,0.440000,0.333333,0.66,1.000000,0.677778,0.766667,0.366667,0.333333,0.386667,0.666667,0.586667


In [7]:
COLS = ['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CLASSES_THEO = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', "Baker's"]

sub.drop(CLASSES_THEO, axis=1, inplace=True)

assert len(sub) == len(sub_theo)
sub = sub.merge(sub_theo[["StudyInstanceUID"] + CLASSES_THEO])
sub = sub[COLS]
assert len(sub) == len(sub_theo)

# Fracture: 50/50 rank average of b1_c1 and the 9-ROI fracture models (fr9a + fr9b), matched by study.
# On the 58 radiologist studies: b1_c1 0.953, fracture models 0.961, blend 0.967. Skipped (b1_c1 kept) if the
# weights have no Fracture entry (the pipeline then writes 0.5 for every study).
theo_frac = sub_theo.set_index("StudyInstanceUID")["Fracture"]
if theo_frac.nunique() > 1:
    b1c1_frac = sub.set_index("StudyInstanceUID")["Fracture"]
    blend = (b1c1_frac.rank(pct=True) + theo_frac.reindex(b1c1_frac.index).rank(pct=True)) / 2
    sub["Fracture"] = sub.StudyInstanceUID.map(blend).values
    print("Fracture: b1_c1 + fracture models (rank average)")
else:
    print("Fracture: b1_c1 only (no fracture models in the weights)")
assert sub[COLS[1:]].notna().all().all()

sub.to_csv('submission.csv', index=False)
sub.head()

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.333333,1.000000,1.000000,0.333333,0.777778,0.082823,0.207248,0.693333,0.666667,1.000000,0.333333,0.413333
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.833333,0.333333,0.666667,0.888889,0.333333,0.433517,0.726805,0.940000,1.000000,0.333333,1.000000,1.000000
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.833333,0.666667,0.333333,0.777778,0.888889,0.279636,0.467260,0.366667,0.333333,0.666667,0.666667,0.586667


In [8]:
!rm -rf src specialists

Done !